<a href="https://colab.research.google.com/github/rirannda/colab-poker/blob/main/colab-poker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ファイブカードドローポーカー**

## **概要**

本プログラムは、コンピュータと対戦できるファイブカードドローポーカーをPythonで実装したものです。プレイヤー1人とCPU3人の合計4人でゲームを行い、各プレイヤーに5枚のカードを配り、ベット・カード交換・再度のベット・勝敗判定という一連の流れを再現しています。

CPUは配られた手札を評価し、役の強さに応じて「チェック」「コール」「レイズ」「フォールド」を選択します。また、役が弱い場合でも一定確率でブラフを行うようにしており、人間らしい判断をするよう工夫しています。カード交換では、役の種類に応じて交換するカードの枚数を変更することで、より自然なプレイになるよう設計しました。

手札の役判定には`treys`ライブラリを利用し、役の比較や勝敗判定を正確に行っています。ゲーム終了時には全員の手札と役を表示し、最も強い役を持つプレイヤーへポット（賭け金）が与えられます。これにより、実際のファイブカードドローポーカーの流れを体験できるプログラムとなっています。

ゲームの流れは以下のようになっています。

```text
カード配布
    ↓
Pre Draw Bet
    ↓
カード交換（Draw）
    ↓
Post Draw Bet
    ↓
Show Down（手札公開）
    ↓
勝敗判定・ポット獲得
```

---

## **操作方法**

1. プログラムを実行すると、プレイヤーとCPU3人に5枚ずつカードが配られます。
2. 最初のベッティング（Pre Draw Bet）が始まります。
3. 画面の案内に従い、番号を入力して行動を選択します。

| 入力 | 行動    |
| -- | ----- |
| 1  | CHECK |
| 2  | RAISE |
| 3  | FOLD  |

4. **RAISE**を選択した場合は、続けてベットするチップ数を入力してください。
5. ベッティング終了後、交換したいカード番号をスペース区切りで入力します。例えば「1 3 5」と入力すると1枚目・3枚目・5枚目を交換します。交換しない場合は「0」を入力してください。
6. CPUもカード交換を行った後、2回目のベッティング（Post Draw Bet）が始まります。
7. 最後に全員の手札と役が公開され、勝敗が表示されます。

### 実行時の注意

* 数字以外を入力すると再入力を求められます。
* ベット額は所持チップ以内で入力してください。
* カード番号は1～5の範囲で入力してください。
* プログラムの実行には`treys`ライブラリが必要です。インストールされていない場合は、事前にインストールしてください。

```bash
pip install treys
```

---

## **制作者のコメント**

今回の制作では、単純にポーカーの役判定を行うだけではなく、CPUが状況に応じて行動を変えるBotを作成した点に力を入れました。役が強い場合には積極的にレイズし、弱い場合にはフォールドするだけでなく、一定確率でブラフを行うようにすることで、毎回異なる展開になるよう工夫しました。また、カード交換では役ごとに交換枚数を変えるようにし、人間のプレイに近い挙動を目指しました。

開発中は、ベット額の管理やポット計算、CPUの行動順など、複数の情報を同時に管理する必要があり、変数の更新タイミングに苦労しました。特に、レイズ後に他のプレイヤーがコールする処理や、フォールドしたプレイヤーを勝敗判定から除外する処理は、何度も動作確認を行いながら修正しました。

一方で、実際のポーカーではベットラウンドが全員の賭け金が揃うまで続くことや、より高度なCPUの戦略など、本プログラムでは簡略化している部分もあります。今後はベット処理の改善や難易度の異なるCPUの追加、GUI化などにも挑戦し、さらに完成度の高いゲームへ発展させたいと考えています。

今回の制作を通して、ゲーム開発ではアルゴリズムだけでなく、ゲームの進行や状態管理が非常に重要であることを学ぶことができました。

---

## **作成時間**

作成時間：**約10時間**

In [ ]:
# ライブラリのインストール
!pip install treys

In [ ]:
# @title
%reset -f
import random
from collections import Counter
from treys import Card, Deck, Evaluator

# カードプールを作成
deck = Deck()

# 各プレイヤーの最初の手札を決定
player_hand = deck.draw(5)
cpu1_hand = deck.draw(5)
cpu2_hand = deck.draw(5)
cpu3_hand = deck.draw(5)

# 各プレイヤーの初期チップを付与
player_chip = 100
cpu1_chip = 100
cpu2_chip = 100
cpu3_chip = 100

def eval_hand(hand):
  evaluator = Evaluator()
  board = [] # ファイブカードドローなので場札はなし

  # 役の強さを評価
  # 1~7462 (1: ロイヤルストレートフラッシュ, 7462: 7-5-4-3-2のスートが異なる役無し)
  score = evaluator.evaluate(board, hand)

  # スコアから役のクラス（ワンペア、フラッシュなど）を取得
  rank_class = evaluator.get_rank_class(score)
  rank_name = evaluator.class_to_string(rank_class)

  # print(f"スコア: {score}")
  # print(f"クラス: {rank_class}")
  # print(f"役の名前: {rank_name}")

  return (score,rank_class,rank_name)


def decide_action(score, rank_class, current_bet):
  # ブラフなどの確率
  random_factor = random.random()
  random_bet = random.random()

  # パターンA: 非常に強い手札（ツーペア以上）
  if rank_class < 8:
    if current_bet == 0:
      # 10%で50,20%で10,70%で20 ベットする
      if random_bet < 0.1:
        bet = 50
      elif random_bet > 0.8:
        bet = 10
      else:
        bet = 20
      return "RAISE", bet
    else:
      # 相手が賭けてきた場合、20%でレイズ、基本はコール
      if random_factor < 0.2:
        if random_bet < 0.2:
          bet = 25
        else:
          bet = 10
        return "RAISE", bet
      else:
        return "CALL"

  # パターンB: ワンペア
  elif rank_class == 8:
    if current_bet == 0:
      # 誰も賭けていなければチェック、30%でレイズ
      return "CHECK" if random_factor < 0.7 else "RAISE", 20
    else:
      # 相手が賭けてきたら基本はコール、20%で降りる
      return "CALL" if random_factor < 0.8 else "FOLD"

  # パターンC: 役なし（ハイカード）
  else:
    if current_bet == 0:
      # 役がないのに、15%の確率で強気にベット(ブラフ)
      if random_factor < 0.15:
        return "RAISE", 20
      return "CHECK"
    else:
      # 相手が賭けてきたら、役がないので基本は降りる,5%でブラフでレイズ
      if random_factor < 0.05:
        return "RAISE", 10
      return "FOLD"

current_bet = 0

player_bet = 0
cpu1_bet = 0
cpu2_bet = 0
cpu3_bet = 0

player_status = None
cpu1_status = None
cpu2_status = None
cpu3_status = None

# プリドローベット

# プレイヤーの番
print("あなたの手札:")
Card.print_pretty_cards(player_hand)
print("\n========================")
print("Pre Draw Bet")
print("========================")
print("あなたの番")
print("選択肢: 1:CHECK, 2:RAISE, 3:FOLD")
pre_choised = False
while pre_choised == False:
  try:
    choice = int(input("\n番号を入力してください: "))
    if choice == 1:
      print("Player: CHECK")
      player_status = "CHECK"
      pre_choised = True
      break
    elif choice == 2:
      pre_choised = True
      player_pre_bet = False
      while player_pre_bet == False:
        try:
          bet = int(input("\n ベットする額を入力してください: "))
          if bet > player_chip:
            print("手持ちのチップ以上に賭けることは出来ません。")
          else:
            player_bet = bet
            player_chip = player_chip - bet
            current_bet = bet
            player_status = "RAISE"
            print(f"Player: RAISE {bet}")
            player_pre_bet = True
        except ValueError:
          print("数値を入力してください。")
    elif choice == 3:
      pre_choised = True
      print("Player: FOLD")
      player_status = "FOLD"
      break
    else:
      print("無効な番号です。リストから選んでください。")
  except ValueError:
      print("数値を入力してください。")


# CPUのターン用関数
def cpu_turn(name, hand, chip, current_bet):

    print(f"\n{name}の番")

    score, rank_class, rank_name = eval_hand(hand)

    result = decide_action(score, rank_class, current_bet)

    if isinstance(result, tuple):
        action, amount = result
    else:
        action = result
        amount = 0

    bet = 0

    if action == "CHECK":
        print(f"{name}: CHECK")

    elif action == "CALL":
        call_amount = current_bet

        if call_amount > chip:
            call_amount = chip

        chip -= call_amount
        bet = call_amount

        print(f"{name}: CALL ({call_amount})")

    elif action == "RAISE":

        total_bet = current_bet + amount

        if total_bet > chip:
            total_bet = chip

        chip -= total_bet
        bet = total_bet
        current_bet = total_bet

        print(f"{name}: RAISE ({total_bet})")

    elif action == "FOLD":
        print(f"{name}: FOLD")

    return chip, bet, action, current_bet

# CPU1
cpu1_chip, cpu1_bet, cpu1_status, current_bet = cpu_turn(
    "CPU1",
    cpu1_hand,
    cpu1_chip,
    current_bet
)

# CPU2
cpu2_chip, cpu2_bet, cpu2_status, current_bet = cpu_turn(
    "CPU2",
    cpu2_hand,
    cpu2_chip,
    current_bet
)

# CPU3
cpu3_chip, cpu3_bet, cpu3_status, current_bet = cpu_turn(
    "CPU3",
    cpu3_hand,
    cpu3_chip,
    current_bet
)

# プレイヤーがコールするか判定
if player_status != "FOLD" and player_bet < current_bet:

    diff = current_bet - player_bet

    print("\nCPUがレイズしました。")
    print(f"コールするには {diff} チップ必要です。")
    print("1:CALL")
    print("2:FOLD")

    while True:
        try:
            choice = int(input("番号を入力してください: "))

            if choice == 1:

                if diff > player_chip:
                    print("チップが足りません。オールインします。")
                    diff = player_chip

                player_chip -= diff
                player_bet += diff
                player_status = "CALL"

                print("Player: CALL")
                break

            elif choice == 2:

                player_status = "FOLD"
                print("Player: FOLD")
                break

            else:
                print("1か2を入力してください。")

        except ValueError:
            print("数字を入力してください。")

# ベット結果表示
print("\n========================")
print("Pre Draw Bet 終了")
print("========================")

print(f"Player : {player_status}  Bet:{player_bet}  Chip:{player_chip}")
print(f"CPU1   : {cpu1_status}  Bet:{cpu1_bet}  Chip:{cpu1_chip}")
print(f"CPU2   : {cpu2_status}  Bet:{cpu2_bet}  Chip:{cpu2_chip}")
print(f"CPU3   : {cpu3_status}  Bet:{cpu3_bet}  Chip:{cpu3_chip}")

pot = player_bet + cpu1_bet + cpu2_bet + cpu3_bet

print(f"\n現在のポット : {pot}")

# プレイヤーのカード交換
print("\n========================")
print("DRAW PHASE")
print("========================")

print("あなたの現在の手札")
Card.print_pretty_cards(player_hand)

print("\n交換したいカード番号を入力してください")
print("例：1 3 5")
print("交換しない場合は0")

while True:

    text = input(">> ")

    if text == "0":
        print("カードを交換しません。")
        break

    try:
        index_list = sorted(
            list(set(map(int, text.split()))),
            reverse=True
        )

        ok = True

        for x in index_list:
            if x < 1 or x > 5:
                ok = False

        if not ok:
            print("1～5で入力してください")
            continue

        for idx in index_list:
            player_hand.pop(idx-1)

        new_cards = deck.draw(len(index_list))
        player_hand.extend(new_cards)

        print("\n交換後の手札")
        Card.print_pretty_cards(player_hand)

        break

    except:
        print("入力形式が違います")
def cpu_draw(hand):

    evaluator = Evaluator()

    score = evaluator.evaluate([], hand)
    rank = evaluator.get_rank_class(score)

    # ツーペア以上は交換しない
    if rank < 8:
        return hand

    ranks = [Card.get_rank_int(c) for c in hand]
    counter = Counter(ranks)

    change = []

    # ワンペア
    if rank == 8:

        pair_rank = None

        for r, c in counter.items():
            if c == 2:
                pair_rank = r

        for i, card in enumerate(hand):
            if Card.get_rank_int(card) != pair_rank:
                change.append(i)

    # ハイカード
    else:

        high = max(ranks)

        for i, card in enumerate(hand):

            if Card.get_rank_int(card) != high:
                change.append(i)

    change.sort(reverse=True)

    for i in change:
        hand.pop(i)

    hand.extend(deck.draw(len(change)))

    return hand

print("\nCPUがカードを交換しています...")

cpu1_hand = cpu_draw(cpu1_hand)
cpu2_hand = cpu_draw(cpu2_hand)
cpu3_hand = cpu_draw(cpu3_hand)

print("カード交換終了")

print("\nあなたの最終手札")
Card.print_pretty_cards(player_hand)

def decide_action_post(score, rank_class, current_bet):

    rand = random.random()
    rand_bet = random.random()

    # ツーペア以上
    if rank_class < 8:

        if current_bet == 0:

            if rand_bet < 0.2:
                return ("RAISE", 50)
            elif rand_bet < 0.7:
                return ("RAISE", 30)
            else:
                return ("RAISE", 20)

        else:

            if rand < 0.3:
                return ("RAISE", 20)
            else:
                return "CALL"

    # ワンペア
    elif rank_class == 8:

        if current_bet == 0:

            if rand < 0.5:
                return "CHECK"
            else:
                return ("RAISE", 20)

        else:

            if rand < 0.9:
                return "CALL"
            else:
                return "FOLD"

    # ハイカード
    else:

        if current_bet == 0:

            if rand < 0.1:
                return ("RAISE", 20)
            else:
                return "CHECK"

        else:

            if rand < 0.1:
                return "CALL"
            else:
                return "FOLD"

def cpu_turn_post(name, hand, chip, current_bet):

    score, rank_class, rank_name = eval_hand(hand)

    result = decide_action_post(score, rank_class, current_bet)

    if isinstance(result, tuple):
        action, amount = result
    else:
        action = result
        amount = 0

    bet = 0

    if action == "CHECK":

        print(f"{name}: CHECK")

    elif action == "CALL":

        call = min(current_bet, chip)

        chip -= call
        bet += call

        print(f"{name}: CALL ({call})")

    elif action == "RAISE":

        total = current_bet + amount

        if total > chip:
            total = chip

        chip -= total
        bet += total
        current_bet = total

        print(f"{name}: RAISE ({total})")

    elif action == "FOLD":

        print(f"{name}: FOLD")

    return chip, bet, action, current_bet

print("\n========================")
print("POST DRAW BET")
print("========================")

current_bet = 0

player_bet = 0
cpu1_bet = 0
cpu2_bet = 0
cpu3_bet = 0

print("あなたの手札")
Card.print_pretty_cards(player_hand)

print("1:CHECK")
print("2:RAISE")
print("3:FOLD")

while True:

    try:

        choice = int(input("番号："))

        if choice == 1:

            player_status = "CHECK"
            print("Player : CHECK")
            break

        elif choice == 2:

            bet = int(input("ベット額："))

            if bet > player_chip:
                print("チップが足りません")
                continue

            player_chip -= bet
            player_bet = bet
            current_bet = bet

            player_status = "RAISE"

            print(f"Player : RAISE {bet}")

            break

        elif choice == 3:

            player_status = "FOLD"

            print("Player : FOLD")

            break

        else:

            print("1～3を入力してください")

    except:

        print("数字を入力してください")

cpu1_chip, cpu1_bet, cpu1_status, current_bet = cpu_turn_post(
    "CPU1",
    cpu1_hand,
    cpu1_chip,
    current_bet
)

cpu2_chip, cpu2_bet, cpu2_status, current_bet = cpu_turn_post(
    "CPU2",
    cpu2_hand,
    cpu2_chip,
    current_bet
)

cpu3_chip, cpu3_bet, cpu3_status, current_bet = cpu_turn_post(
    "CPU3",
    cpu3_hand,
    cpu3_chip,
    current_bet
)

if player_status != "FOLD" and player_bet < current_bet:

    diff = current_bet - player_bet

    print(f"\n追加で {diff} チップ必要です")

    print("1:CALL")
    print("2:FOLD")

    while True:

        try:

            c = int(input())

            if c == 1:

                diff = min(diff, player_chip)

                player_chip -= diff
                player_bet += diff

                player_status = "CALL"

                print("Player : CALL")

                break

            elif c == 2:

                player_status = "FOLD"

                print("Player : FOLD")

                break

        except:

            print("数字を入力してください")

pot += player_bet
pot += cpu1_bet
pot += cpu2_bet
pot += cpu3_bet

print("\n現在のポット")

print(pot)

print("SHOW DOWN")

print("\nPlayer")
Card.print_pretty_cards(player_hand)

print("\nCPU1")
Card.print_pretty_cards(cpu1_hand)

print("\nCPU2")
Card.print_pretty_cards(cpu2_hand)

print("\nCPU3")
Card.print_pretty_cards(cpu3_hand)

player_score, _, player_rank = eval_hand(player_hand)
cpu1_score, _, cpu1_rank = eval_hand(cpu1_hand)
cpu2_score, _, cpu2_rank = eval_hand(cpu2_hand)
cpu3_score, _, cpu3_rank = eval_hand(cpu3_hand)

scores = {}

if player_status != "FOLD":
    scores["Player"] = player_score

if cpu1_status != "FOLD":
    scores["CPU1"] = cpu1_score

if cpu2_status != "FOLD":
    scores["CPU2"] = cpu2_score

if cpu3_status != "FOLD":
    scores["CPU3"] = cpu3_score

winner = min(scores, key=scores.get)

print("\n========================")
print("結果")
print("========================")

print(f"勝者 : {winner}")

if winner == "Player":
    player_chip += pot

elif winner == "CPU1":
    cpu1_chip += pot

elif winner == "CPU2":
    cpu2_chip += pot

elif winner == "CPU3":
    cpu3_chip += pot

print("\n========================")
print("所持チップ")
print("========================")

print(f"Player : {player_chip}")
print(f"CPU1   : {cpu1_chip}")
print(f"CPU2   : {cpu2_chip}")
print(f"CPU3   : {cpu3_chip}")



あなたの手札:
 [5♠],[Q♠],[3♣],[J♥],[T♦] 

Pre Draw Bet
あなたの番
選択肢: 1:CHECK, 2:RAISE, 3:FOLD

番号を入力してください: 1
Player: CHECK

CPU1の番
CPU1: CHECK

CPU2の番
CPU2: RAISE (20)

CPU3の番
CPU3: FOLD

CPUがレイズしました。
コールするには 20 チップ必要です。
1:CALL
2:FOLD
番号を入力してください: 1
Player: CALL

Pre Draw Bet 終了
Player : CALL  Bet:20  Chip:80
CPU1   : CHECK  Bet:0  Chip:100
CPU2   : RAISE  Bet:20  Chip:80
CPU3   : FOLD  Bet:0  Chip:100

現在のポット : 40

DRAW PHASE
あなたの現在の手札
 [5♠],[Q♠],[3♣],[J♥],[T♦] 

交換したいカード番号を入力してください
例：1 3 5
交換しない場合は0
>> 1 3

交換後の手札
 [Q♠],[J♥],[T♦],[A♦],[7♦] 

CPUがカードを交換しています...
カード交換終了

あなたの最終手札
 [Q♠],[J♥],[T♦],[A♦],[7♦] 

POST DRAW BET
あなたの手札
 [Q♠],[J♥],[T♦],[A♦],[7♦] 
1:CHECK
2:RAISE
3:FOLD
番号：3
Player : FOLD
CPU1: CHECK
CPU2: RAISE (20)
CPU3: FOLD

現在のポット
60
SHOW DOWN

Player
 [Q♠],[J♥],[T♦],[A♦],[7♦] 

CPU1
 [K♥],[A♣],[7♠],[3♦],[T♠] 

CPU2
 [K♦],[K♣],[4♠],[4♥],[J♦] 

CPU3
 [A♠],[6♥],[3♠],[9♣],[K♠] 

結果
勝者 : CPU2

所持チップ
Player : 80
CPU1   : 100
CPU2   : 120
CPU3   : 100
